# Traffic Data as Images and Video

**Goal**: Working with image and vide data for object detection. The focus is on how video data can be converted to images, understanding bounding boxes for object classification, and using XML annotations to represent those bounding boxes. 

**Objectives**

- Load and organize the project dataset, seperating images and their corresponding XML annotations. 

- Extract frames from a video at regular intervals.

- Parse XML annotations. 

- Visualize bounding boxes on image data.

In [1]:
import sys
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt 
from IPython.display import Video
import torch
import torchvision
from torchvision import transforms
from torchvision.io import read_image
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import draw_bounding_boxes, make_grid

In [2]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("CV2 version : ", cv2.__version__)
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)

Platform: win32
Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
---
CV2 version :  4.11.0
torch version :  2.5.1+cpu
torchvision version :  0.20.1+cpu


**Exploring Data**

- The project uses the  [Dhaka AI dataset](https://www.kaggle.com/datasets/rifat963/dhakaai-dhaka-based-traffic-detection-dataset?resource=download-directory), which contains images of vehicles in urban traffic scenes from Dhaka, Bangladesh. This dataset is particularly interesting for computer vision as it captures the unique characteristics of Dhaka's busy streets, including a diverse mix of vehicle types and dense traffic conditions.

- We use the dataset for object detection which is a more complex task than image classification. Object detection identifies specific objects within an image (e.g., cars, buses, motorcycles), determines the precise location of these objects, and draws a bounding box around each detected object.

In [3]:
dhaka_image_dir = Path("data_images", "train")

print("Data directory:", dhaka_image_dir)

Data directory: data_images\train


In [4]:
dhaka_files = list(dhaka_image_dir.iterdir())
dhaka_files[-5:]

[WindowsPath('data_images/train/Dipto_514.xml'),
 WindowsPath('data_images/train/Dipto_515.jpg'),
 WindowsPath('data_images/train/Dipto_515.xml'),
 WindowsPath('data_images/train/Dipto_516.jpg'),
 WindowsPath('data_images/train/Dipto_516.xml')]

- Counting the file extensions by type:

In [5]:
file_extension_counts = Counter(Path(file).suffix for file in dhaka_files)

for extension, count in file_extension_counts.items():
    print(f"Files with extension {extension}: {count}")

Files with extension .jpg: 788
Files with extension .xml: 813
Files with extension .jpeg: 2
Files with extension .png: 12
Files with extension .JPG: 11


**Seperating images and bounding boxes data**

In [6]:
images_dir = dhaka_image_dir / "images"
annotations_dir = dhaka_image_dir / "annotations"

images_dir.mkdir(exist_ok=True)
annotations_dir.mkdir(exist_ok=True)

Moving files to appropriate directories based on extensions.

In [7]:
for file in dhaka_files:
    if file.suffix.lower() in (".jpg", ".jpeg", ".png"):
        target_dir = images_dir
    elif file.suffix.lower() == ".xml":
        target_dir = annotations_dir
    file.rename(target_dir / file.name)

Check number of images and annotations:

In [8]:
images_files = list(images_dir.iterdir())
annotations_files = list(annotations_dir.iterdir())

assert len(images_files) == len(annotations_files)

**Annotations**

- Annotations are the labels for the data. Each image has an annotation that contains the coordinates and type of object for each bounding bix in a given image.

- The annotations are stored as XML, which is a way to store structured documents. 

- The ```<annotation>``` tag is the root element, containing all the information about this particular image annotation. The tags within store other information such as ```<folder>```. 

- The most important tag for the current project is the ```<object>```. It describes an object detected in the image, this associated image contains a "bus". 

- The tag ```<bndbox>``` is the bounding box information. 

- There are the coordinates of a rectangle surrounding the bus in the image (in pixels): ```<xmin>``` is the left edge, ```<ymin>``` is the top edge, ```<xmax>``` is the right edge, and ```<ymax>``` is the bottom edge.

In [16]:
xml_filepath = annotations_dir / "01.xml"
!head -n 25 $xml_filepath

'head' is not recognized as an internal or external command,
operable program or batch file.
